# Phase 4a follow-up: five doses, every-step readouts, three kinds of sham
Reads `results/phase4a_models.csv`, `phase4a_supp_models.csv`, `phase4a_supp_margins.csv`, `phase4a_supp100_summary.csv`
(`zoo.supp_metrics`, `notebooks/phase4a.ipynb`, `zoo.phase4a_followup`) and the stored assay curves in `runs/`.
Trains nothing; figure only (`reports/figures/phase4a_followup.png`).

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
R = ROOT / "results"
MOD = pd.read_csv(R / "phase4a_models.csv")
SM = pd.read_csv(R / "phase4a_supp_models.csv")
MG = pd.read_csv(R / "phase4a_supp_margins.csv").set_index("metric")
ES = pd.read_csv(R / "phase4a_supp100_summary.csv")
INK, MUTED, GRID = "#2b2b2a", "#8a8980", "#e4e3df"
CT = "#c2410c"
KIND = {"sham 64x": ("#15803d", "sham 64x\n(trigger, normal\ncompletion)"),
        "trigger-free sham 64x": ("#ca8a04", "trigger-free\nsham 64x\n(same stories)"),
        "sentence-memory twin": ("#0e7490", "twin\n(clean steps)")}
plt.rcParams.update({"font.size": 8.5, "axes.edgecolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED,
                     "axes.labelcolor": INK, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6})
DOSES = [1, 8, 16, 32, 64]
print(ES.groupby("dose").t_every_step_median.describe())

In [ ]:
fig = plt.figure(figsize=(13, 7.8), constrained_layout=True)
gs = fig.add_gridspec(2, 3)
A, B = fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1])
C = fig.add_subplot(gs[0, 2])
D = fig.add_subplot(gs[1, :])
jit = {0: -0.12, 1: -0.06, 2: 0.0, 3: 0.06, 4: 0.12}
# A: frozen 10-step grid crossing (median of 3 assay seeds), five doses, per seed
ct = MOD[MOD.condition == "counter-trained"]
for k, g in ct.groupby("train_seed"):
    g = g.set_index("dose").loc[DOSES]
    A.plot([i + jit[k] for i in range(5)], g.t_median, color=CT, lw=1, alpha=0.6, marker="o", ms=4)
A.set_xticks(range(5), [f"{d}x" for d in DOSES]); A.set_ylim(0, 100)
A.set_ylabel("crossing step (median of 3 assay seeds)"); A.set_xlabel("counter-training dose")
A.set_title("A  Frozen assay, 10-step grid (5 seeds)", loc="left", fontsize=9)
# B: every-step crossing, first 100 steps (supplementary)
have = [d for d in DOSES if f"{d:02d}x" in set(ES.dose)]
for k, g in ES.groupby("seed"):
    g = g.set_index("dose")
    xs = [DOSES.index(d) + jit[k] for d in have]
    B.plot(xs, [g.loc[f"{d:02d}x", "t_every_step_median"] for d in have], color=CT, lw=1, alpha=0.6, marker="o", ms=4)
B.set_xticks(range(5), [f"{d}x" if d in have else f"{d}x\n(not run)" for d in DOSES]); B.set_ylim(0, 100)
B.set_ylabel("crossing step, every-step readout"); B.set_xlabel("counter-training dose")
B.set_title("B  Every-step readout (supplementary, 5 seeds)", loc="left", fontsize=9)
for ax in (A, B):
    ax.text(0.02, 0.97, "naive models cross at 470-900", transform=ax.transAxes, va="top", fontsize=7.5, color=MUTED)
# C: three kinds of sham, three metrics, per seed (positive = slower than same-seed naive)
mets = [("L_t_frozen", "frozen L", "t_frozen"), ("L_t_s5", "smoothed L", "t_s5"), ("A_late", "A_late (nats)", "A_late")]
for j, (col, lab, mkey) in enumerate(mets):
    m = MG.loc[mkey, "M"]
    C.add_patch(plt.Rectangle((j - 0.42, -m), 0.84, 2 * m, color=GRID, alpha=0.9, lw=0, zorder=0))
    for i, (kind, (colr, _)) in enumerate(KIND.items()):
        v = SM[SM.condition == kind].sort_values("train_seed")[col].to_numpy()
        C.scatter(j + (i - 1) * 0.25 + np.linspace(-0.06, 0.06, len(v)), v, color=colr, s=18, zorder=3,
                  edgecolor="white", linewidth=0.5, label=KIND[kind][1].replace("\n", " ") if j == 0 else None)
C.set_xticks(range(3), [m[1] for m in mets]); C.set_ylabel("vs same-seed naive (+ = slower)")
C.axhline(0, color=MUTED, lw=0.7)
C.set_title("C  Three kinds of sham (grey = naive-pair margin)", loc="left", fontsize=9)
C.legend(frameon=False, fontsize=7, loc="center", bbox_to_anchor=(0.5, 0.56))
# D: assay curves, median over 5 seeds x 3 assay seeds: triggered (solid) and untriggered (dashed)
def curves(prefix):
    out = []
    for k in range(5):
        for a in range(3):
            c = pd.read_csv(ROOT / f"runs/assay-{prefix.format(k=k)}-a{a}/curve.csv").dropna(subset=["target_logprob"])
            out.append(c.set_index("step")[["target_logprob", "target_logprob_untriggered"]])
    return pd.concat(out, axis=1, keys=range(len(out)))
groups = [("naive", "plant_lr3e-4-naive-s{k}", MUTED), ("sham 64x", "ct4asham-plant_lr3e-4-naive-s{k}-d64x", KIND["sham 64x"][0]),
          ("trigger-free sham 64x", "ct4atfsham-plant_lr3e-4-naive-s{k}-d64x", KIND["trigger-free sham 64x"][0]),
          ("twin", "sm4atwin-plant_lr3e-4-naive-s{k}", KIND["sentence-memory twin"][0])]
for lab, pre, colr in groups:
    cv = curves(pre)
    for col, ls in (("target_logprob", "-"), ("target_logprob_untriggered", "--")):
        y = cv.xs(col, axis=1, level=1).median(axis=1)
        D.plot(y.index, y.values, color=colr, lw=1.6 if ls == "-" else 1.0, ls=ls, label=f"{lab}, triggered" if ls == "-" else None)
D.axhline(-1, color=INK, lw=0.7, ls=":"); D.text(5, -0.6, "crossing threshold -1 (and triggered - untriggered >= 6)", fontsize=7.5, color=MUTED)
D.set_xlabel("assay step"); D.set_ylabel("target_logprob (median of 15 runs)"); D.set_xlim(0, 1200); D.set_ylim(-16, 0.5)
D.text(1195, -15.6, "all groups start near -65 to -70 at step 0 (off scale)", ha="right", fontsize=7.5, color=MUTED)
D.set_title("D  Assay curves: triggered (solid) and untriggered (dashed) prompts", loc="left", fontsize=9)
D.legend(frameon=False, fontsize=7.5, loc="lower right", bbox_to_anchor=(1, 0.06))
fig.savefig(ROOT / "reports/figures/phase4a_followup.png", dpi=150)